In [178]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import style
style.use('ggplot')

In [179]:
horas = ['9-10', '10-11', '11-12', '12-13', '13-14', '14-15', '15-16', '16-17', '17-18']
lambdas = [50, 55, 45, 35, 20, 35, 40, 60, 10]

df = pd.DataFrame({
    'hora':horas,
    'lambda':lambdas
})
df

,hora,lambda
0,9-10,50
1,10-11,55
2,11-12,45
3,12-13,35
4,13-14,20
5,14-15,35
6,15-16,40
7,16-17,60
8,17-18,10


In [180]:
# Mezcla de operaciones
# probs = [0.33, 0.28, 0.25, 0.07, 0.03]
# tiempos = [4, 2, 2, 4, 8]

In [181]:
def calcular_mu(probs, tiempos):
    """
    Calcula la tasa de servicio μ (clientes por hora)
    a partir de probabilidades y tiempos promedio (en minutos).
    """
    
    # Probabilidad conocida
    p_conocida = sum(probs)
    
    # Tiempo promedio ponderado conocido
    t_prom_conocido = sum(p * t for p, t in zip(probs, tiempos))
    
    # Probabilidad restante
    p_rest = 1 - p_conocida
    
    # Tiempo promedio del resto (ajuste proporcional)
    t_resto = t_prom_conocido / p_conocida
    
    # Tiempo promedio total
    t_prom_min = t_prom_conocido + p_rest * t_resto
    
    # Convertimos a clientes por hora
    mu = 60 / t_prom_min
    
    return mu


In [182]:

def metricas_mmc(lam, mu, c):

    r = lam / mu
    rho = lam / (c * mu)

    if rho >= 1:
        return None, None, None, None, None, None, None, None

    suma = sum((r**k)/math.factorial(k) for k in range(c))
    termino_final = (r**c)/(math.factorial(c)*(1-rho))
    P0 = 1/(suma + termino_final)

    Lq = (P0 * (r**c) * rho) / (math.factorial(c) * (1-rho)**2)

    
    Wq_h = Lq / lam
    Ws_h = Wq_h + (1/mu)
    Ls = lam * Ws_h
    Wq_m = Wq_h * 60
    Ws_m = Ws_h * 60

    return rho, P0, Lq, Wq_h, Ws_h, Ls, Wq_m, Ws_m

In [183]:
# frecuencia de operaciones 9-10
probs09_10 = [0.33, 0.28, 0.25, 0.07, 0.03]
tiempos09_10 = [4, 2, 2, 4, 8]

# frecuencia de operaciones 10-11
probs10_11 = [0.34, 0.29, 0.20, 0.06, 0.02]
tiempos10_11 = [4, 2, 2, 4, 8]

# frecuencia de operaciones 11-12
probs11_12 = [0.20, 0.33, 0.15, 0.17, 0.13]
tiempos11_12 = [4, 2, 2, 4, 8]

# frecuencia de operaciones 12-13
probs12_13 = [0.23, 0.34, 0.22, 0.11, 0.04]
tiempos12_13 = [4, 2, 2, 4, 8]

# frecuencia de operaciones 13-14
probs13_14 = [0.33, 0.18, 0.22, 0.04, 0.13]
tiempos13_14 = [4, 2, 2, 4, 8]

# frecuencia de operaciones 14-15
probs14_15 = [0.13, 0.38, 0.22, 0.02, 0.14]
tiempos14_15 = [4, 2, 2, 4, 8]

# frecuencia de operaciones 15-16
probs15_16 = [0.21, 0.21, 0.33, 0.17, 0.02]
tiempos15_16 = [4, 2, 2, 4, 8]

# frecuencia de operaciones 16-17
probs16_17 = [0.20, 0.20, 0.10, 0.10, 0.11]
tiempos16_17 = [4, 2, 2, 4, 8]

# frecuencia de operaciones 17-18
probs17_18 = [0.23, 0.20, 0.05, 0.01, 0.33]
tiempos17_18 = [4, 2, 2, 4, 8]

tipos = ["Op1", "Op2", "Op3", "Op4", "Op5"]

df_operaciones = pd.DataFrame({
    "Operacion": tipos,
    "Tiempo_min": [3, 3, 3, 4.8, 20],
    "9-10": probs09_10,
    "10-11": probs10_11,
    "11-12": probs11_12,
    "12-13": probs12_13,
    "13-14": probs13_14,
    "14-15": probs14_15,
    "15-16": probs15_16,
    "16-17": probs16_17,
    "17-18": probs17_18
})

df_operaciones

,Operacion,Tiempo_min,9-10,10-11,11-12,12-13,13-14,14-15,15-16,16-17,17-18
0,Op1,3.0,0.33,0.34,0.20,0.23,0.33,0.13,0.21,0.20,0.23
1,Op2,3.0,0.28,0.29,0.33,0.34,0.18,0.38,0.21,0.20,0.20
2,Op3,3.0,0.25,0.20,0.15,0.22,0.22,0.22,0.33,0.10,0.05
3,Op4,4.8,0.07,0.06,0.17,0.11,0.04,0.02,0.17,0.10,0.01
4,Op5,20.0,0.03,0.02,0.13,0.04,0.13,0.14,0.02,0.11,0.33


In [184]:
# df["mu"] = df.apply(lambda row: calcular_mu(probs, tiempos), axis=1)
mu_por_hora = {}

tiempos = df_operaciones["Tiempo_min"].values

for hora in df_operaciones.columns[2:]:  # saltamos Operacion y Tiempo_min
    probs = df_operaciones[hora]
    mu_por_hora[hora] = calcular_mu(probs, tiempos)



df["mu"] = df["hora"].map(mu_por_hora)

df["s"] = 5


df[["rho","P0","Lq","Wq_h","Ws_h","Ls", "Wq_m", "Ws_m"]] = df.apply(
    lambda row: metricas_mmc(row["lambda"], row["mu"], row["s"]),
    axis=1,
    result_type="expand"
)

df


,hora,lambda,mu,s,rho,P0,Lq,Wq_h,Ws_h,Ls,Wq_m,Ws_m
0,9-10,50,16.382253,5,0.610417,0.043997,0.390527,0.007811,0.068852,3.442611,0.468633,4.131133
1,10-11,55,17.180617,5,0.640256,0.037095,0.514190,0.009349,0.067554,3.715472,0.560934,4.053242
2,11-12,45,10.777126,5,0.835102,0.009724,3.158886,0.070197,0.162987,7.334396,4.211848,9.779195
3,12-13,35,15.251487,5,0.458972,0.099223,0.082520,0.002358,0.067925,2.377378,0.141462,4.075505
4,13-14,20,10.839020,5,0.369037,0.157230,0.025979,0.001299,0.093558,1.871164,0.077936,5.613492
5,14-15,35,10.499410,5,0.666704,0.031745,0.653559,0.018673,0.113917,3.987080,1.120387,6.834994
6,15-16,40,16.272360,5,0.491631,0.083699,0.119088,0.002977,0.064431,2.577244,0.178632,3.865866
7,16-17,60,10.191388,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,17-18,10,6.083086,5,0.328780,0.192720,0.014070,0.001407,0.165797,1.657973,0.084422,9.947836


In [185]:
def servidores_optimos(lam, mu, umbral_wq_m=5, rho_max=0.85, c_max=20):

    for c in range(1, c_max + 1):

        metrics = metricas_mmc(lam, mu, c)

        if metrics[0] is None:
            continue

        rho = metrics[0]
        Wq_m = metrics[6]

        if rho < rho_max and Wq_m <= umbral_wq_m:

            return (c,) + metrics   # une servidores con todas las métricas

    return (None,)*9


df[[
    "s_o",
    "rho_o",
    "P0_o",
    "Lq_o",
    "Wq_h_o",
    "Ws_h_o",
    "Ls_o",
    "Wq_m_o",
    "Ws_m_o"
]] = df.apply(
    lambda row: servidores_optimos(row["lambda"], row["mu"]),
    axis=1,
    result_type="expand"
)
df

,hora,lambda,mu,s,rho,P0,Lq,Wq_h,Ws_h,Ls,...,Ws_m,s_o,rho_o,P0_o,Lq_o,Wq_h_o,Ws_h_o,Ls_o,Wq_m_o,Ws_m_o
0,9-10,50,16.382253,5,0.610417,0.043997,0.390527,0.007811,0.068852,3.442611,...,4.131133,4.0,0.763021,0.034837,1.711327,0.034227,0.095268,4.763411,2.053593,5.716093
1,10-11,55,17.180617,5,0.640256,0.037095,0.514190,0.009349,0.067554,3.715472,...,4.053242,4.0,0.800321,0.027241,2.392829,0.043506,0.101711,5.594111,2.610359,6.102667
2,11-12,45,10.777126,5,0.835102,0.009724,3.158886,0.070197,0.162987,7.334396,...,9.779195,5.0,0.835102,0.009724,3.158886,0.070197,0.162987,7.334396,4.211848,9.779195
3,12-13,35,15.251487,5,0.458972,0.099223,0.082520,0.002358,0.067925,2.377378,...,4.075505,3.0,0.764953,0.068977,1.923724,0.054964,0.120531,4.218582,3.297812,7.231854
4,13-14,20,10.839020,5,0.369037,0.157230,0.025979,0.001299,0.093558,1.871164,...,5.613492,3.0,0.615062,0.137597,0.598018,0.029901,0.122160,2.443204,1.794055,7.329611
5,14-15,35,10.499410,5,0.666704,0.031745,0.653559,0.018673,0.113917,3.987080,...,6.834994,5.0,0.666704,0.031745,0.653559,0.018673,0.113917,3.987080,1.120387,6.834994
6,15-16,40,16.272360,5,0.491631,0.083699,0.119088,0.002977,0.064431,2.577244,...,3.865866,3.0,0.819385,0.049540,3.080437,0.077011,0.138465,5.538593,4.620656,8.307890
7,16-17,60,10.191388,5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,7.0,0.841046,0.001883,3.048815,0.050814,0.148936,8.936139,3.048815,8.936139
8,17-18,10,6.083086,5,0.328780,0.192720,0.014070,0.001407,0.165797,1.657973,...,9.947836,3.0,0.547967,0.177523,0.352489,0.035249,0.199639,1.996392,2.114936,11.978351


In [186]:
import simpy
import random
import numpy as np

def simular_agencia(lam, mu, ventanillas, tiempo_sim=480):

    esperas = []

    def cliente(env, ventanillas):

        llegada = env.now

        with ventanillas.request() as req:
            yield req

            espera = env.now - llegada
            esperas.append(espera)

            servicio = random.expovariate(mu)
            yield env.timeout(servicio)

    def generador(env, ventanillas):

        while True:
            yield env.timeout(random.expovariate(lam))
            env.process(cliente(env, ventanillas))

    env = simpy.Environment()
    recurso = simpy.Resource(env, capacity=ventanillas)

    env.process(generador(env, recurso))

    env.run(until=tiempo_sim)

    return {
        "clientes": len(esperas),
        "espera_promedio": np.mean(esperas),
        "p95": np.percentile(esperas,95)
    }

In [187]:
simulaciones = []

for _, row in df.iterrows():

    lam = row["lambda"]/60
    mu = row["mu"]/60

    for v in range(3,10):

        resultado = simular_agencia(lam, mu, v)

        simulaciones.append({
            "hora": row["hora"],
            "ventanillas": v,
            "espera_promedio": resultado["espera_promedio"],
            "espera_p95": resultado["p95"],
            "clientes": resultado["clientes"]
        })

df_sim = pd.DataFrame(simulaciones)

df_sim

,hora,ventanillas,espera_promedio,espera_p95,clientes
0,9-10,3,10.393778,26.900462,398
1,9-10,4,2.205780,6.532162,407
2,9-10,5,0.323714,2.133157,392
3,9-10,6,0.079184,0.631556,395
4,9-10,7,0.003485,0.000000,385
...,...,...,...,...,...
58,17-18,5,0.000000,0.000000,70
59,17-18,6,0.000000,0.000000,69
60,17-18,7,0.000000,0.000000,74
61,17-18,8,0.000000,0.000000,79
